### Factor Decomposition for an S&P Portfolio

In [4]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

from src import data, factors, plotting

spy_px, spy_ret = data.get_market()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
plotting.plot_stock(prices, "AAPL")
plotting.plot_vs_market(prices, spy_px, "AAPL")

### Single-Factor Model: A Naive Look at Idiosyncratic Returns

We decompose a stock's daily return into a market component and a stock-specific component:

$$
r_{i,t} = \alpha_i + \beta_i \, r_{m,t} + \varepsilon_{i,t}
$$

- $r_{i,t}$: return of stock $i$ on day $t$
- $r_{m,t}$: market (SPY) return on day $t$
- $\beta_i = \dfrac{\mathrm{Cov}(r_i, r_m)}{\mathrm{Var}(r_m)}$: sensitivity to the market
- $\alpha_i$: average return not explained by the market
- $\varepsilon_{i,t}$: residual, the **idiosyncratic return**

The idiosyncratic return is what's left after stripping out the market:

$$
\varepsilon_{i,t} = r_{i,t} - \alpha_i - \beta_i \, r_{m,t}
$$

This also splits total variance into systematic and idiosyncratic parts (the residual is uncorrelated with the market by construction):

$$
\mathrm{Var}(r_i) = \beta_i^2 \, \mathrm{Var}(r_m) + \mathrm{Var}(\varepsilon_i)
$$

**Why it's naive:** the market is the only factor. Anything driven by sector or style (size, momentum, volatility) gets lumped into $\varepsilon$, so this overstates true idiosyncratic risk. A multi-factor model peels those layers off next.

In [12]:
plotting.plot_regression(prices, spy_px, "AAPL")

In [13]:
qqq_px, qqq_ret = data.get_market("QQQ")
plotting.plot_regression(prices, qqq_px, "AAPL", market_name="QQQ")

In [14]:
iwm_px, _ = data.get_market("IWM")   # Russell 2000 (small caps)
plotting.plot_regression(prices, iwm_px, "AAPL", market_name="IWM")

### Two-Factor Model: Market + Sector

Extend the single-factor model with a sector return (e.g. XLK for Apple):

$$
r_{i,t} = \alpha_i + \beta_m \, r_{m,t} + \beta_s \, r_{s,t} + \varepsilon_{i,t}
$$

**Problem: collinearity.** Sector and market returns are highly correlated ($\rho \approx 0.85$ to $0.9$ for XLK vs SPY), so the two betas are unstable and hard to interpret.

**Fix: orthogonalize the sector against the market.** First strip the market out of the sector:

$$
r_{s,t} = a + b \, r_{m,t} + u_t
$$

Then regress the stock on the market and the sector residual $u_t$:

$$
r_{i,t} = \alpha_i + \beta_m \, r_{m,t} + \beta_s \, u_t + \varepsilon_{i,t}
$$

- $u_t$: the sector-specific move, i.e. what the sector did beyond the market
- $\beta_m$: identical to the single-factor beta, since $u$ is uncorrelated with $r_m$
- $\beta_s$: sensitivity to the sector-specific move
- $\Delta R^2$ vs the single-factor model: variance explained by the sector on top of the market

**Caveat: self-inclusion.** Apple is a large weight in XLK, so part of the "sector" is Apple itself, which inflates $\beta_s$ and $R^2$. Cleaner proxies are an equal-weight sector ETF or the average return of the other stocks in the sector, excluding the one being modeled.

___

### Orthogonalization: The Idea

Treat each return series as a vector, one entry per day. Two series that move together point in similar directions. That overlap is what makes their effects hard to separate in a regression.

Orthogonalizing $s$ against $m$ splits $s$ into two pieces:

$$
s = \underbrace{b \, m}_{\text{part explained by } m} + \underbrace{u}_{\text{part unrelated to } m}
$$

- $b \, m$ is the **projection** of $s$ onto $m$: the "shadow" $s$ casts in the market's direction, with $b = \dfrac{\mathrm{Cov}(s, m)}{\mathrm{Var}(m)}$
- $u = s - b \, m$ is what's left, and by construction $\mathrm{Cov}(u, m) = 0$

Geometrically, $u$ is perpendicular to $m$: it carries no information about the market at all.

**Why it helps:** the two regressors $m$ and $u$ no longer overlap, so each coefficient is cleanly identified. The market beta is unchanged from the single-factor model, and the sector coefficient captures only what's new, i.e. the sector's own move with the market stripped out.

**The catch:** the result depends on order. Orthogonalizing the sector against the market gives the market "first claim" on the shared variance. Reversing the order would credit that shared variance to the sector instead.
___

### Adding More Factors: Sequential Orthogonalization (Gram-Schmidt)

To add a third factor that may overlap with the first two (e.g. a size proxy like IWM), regress it on **all existing factors at once** and keep the residual:

$$
r_{\text{new},t} = c + d_1 \, r_{m,t} + d_2 \, u_t + v_t
$$

Then add $v_t$ to the stock regression:

$$
r_{i,t} = \alpha_i + \beta_m \, r_{m,t} + \beta_s \, u_t + \beta_v \, v_t + \varepsilon_{i,t}
$$

- $v_t$: the part of the new factor unexplained by market and sector
- $\beta_m, \beta_s$: unchanged from the two-factor model, since $v$ is uncorrelated with both
- $\beta_v$: sensitivity to what's genuinely new in the third factor
- $\Delta R^2$: extra variance explained by the new factor

Because $m$ and $u$ are already orthogonal, regressing on them jointly or one at a time gives the same result. With correlated factors, only the joint regression gives a clean residual.

### Key point: orthogonalization changes interpretation, not fit

Regressing on raw SPY, XLK and IWM together gives the **same** $\varepsilon$ and **same** $R^2$ as the orthogonalized version, since both span the same space. What changes is **who gets credit for shared variance**:

1. Market gets first claim
2. Sector gets what's left beyond market
3. New factor gets what's left beyond both

Order is therefore a modeling choice: put the most fundamental factor first, and each later factor is measured as an increment.

- **If you care about the idiosyncratic residual**, orthogonalization doesn't matter.
- **If you care about clean, stable, interpretable betas**, it does.

In [17]:
xlk_px, _ = data.get_market("XLK")
plotting.plot_vs_market(prices, xlk_px, "AAPL", market_name="XLK")

In [18]:
plotting.plot_two_factor(prices, spy_px, xlk_px, "AAPL")

In [22]:
# market only
plotting.plot_actual_vs_fitted(prices, {"SPY": spy_px}, "AAPL")

In [23]:
# market, then XLK orthogonalized to market
plotting.plot_actual_vs_fitted(prices, {"SPY": spy_px, "XLK": xlk_px}, "AAPL")

___

### Simple 12-Month Momentum
___

In [26]:
mom = factors.momentum(prices)
z_mom = factors.standardize(mom)
res = factors.cs_regression(returns, z_mom)

plotting.plot_factor_returns(res["factor_ret"])

In [25]:
plotting.plot_cs_r2(res["r2"])

___

### Simple Volatility Factor
___

In [29]:
import numpy as np

vol = factors.volatility(returns)
z_vol = factors.standardize(np.log(vol))
res_vol = factors.cs_regression(returns, z_vol)

plotting.plot_factor_returns(res_vol["factor_ret"], name="Volatility (63d)")

In [30]:
plotting.plot_cs_r2(res_vol["r2"], name="Volatility (63d)")

In [32]:
plotting.plot_exposure_corr(z_mom, z_vol)

In [40]:
import pandas as pd

res_joint = factors.cs_regression_multi(returns, {"Momentum": z_mom, "Volatility": z_vol})

plotting.plot_factor_compare(
    {"Momentum (isolated)": res["factor_ret"], "Momentum (joint)": res_joint["Momentum"]},
    title="Momentum: isolated vs controlling for volatility",
)

In [41]:
plotting.plot_factor_compare(
    {"Volatility (isolated)": res_vol["factor_ret"], "Volatility (joint)": res_joint["Volatility"]},
    title="Volatility: isolated vs controlling for momentum",
)

___

### Simple Short-Term Reversal
___

In [42]:
rev = factors.reversal(prices)
z_rev = factors.standardize(rev)
res_rev = factors.cs_regression(returns, z_rev)

plotting.plot_factor_returns(res_rev["factor_ret"], name="Reversal (1m)")

In [43]:
plotting.plot_cs_r2(res_rev["r2"], name="Reversal (1m)")

In [45]:
res_all = factors.cs_regression_multi(
    returns, {"Momentum": z_mom, "Volatility": z_vol, "Reversal": z_rev}
)

plotting.plot_factor_compare(
    {
        "Momentum (isolated)": res["factor_ret"],
        "Momentum | Vol": res_joint["Momentum"],
        "Momentum | Vol + Rev": res_all["Momentum"],
    },
    title="Momentum: isolated vs controlling for other factors",
)

In [48]:
r2 = pd.DataFrame({
    "Momentum only": res["r2"],
    "Mom + Vol": res_joint["r2"],
    "Mom + Vol + Rev": res_all["r2"],
}).agg(["mean", "median"]).mul(100).T
r2.columns = ["Mean R² (%)", "Median R² (%)"]

plotting.plot_table(
    r2, title="Cross-sectional R² by model", index_name="Model",
    caption="Daily cross-sectional regressions of stock returns on lagged standardized exposures.",
)

___

### Simple Beta
___

In [49]:
beta = factors.beta_exposure(returns, spy_ret)
z_beta = factors.standardize(beta)
res_beta = factors.cs_regression(returns, z_beta)

plotting.plot_factor_returns(res_beta["factor_ret"], name="Beta (252d)")

In [50]:
plotting.plot_cs_r2(res_beta["r2"], name="Beta (252d)")

In [51]:
plotting.plot_exposure_corr(z_beta, z_vol, "Beta", "Volatility")

In [52]:
res_4 = factors.cs_regression_multi(
    returns, {"Momentum": z_mom, "Volatility": z_vol, "Reversal": z_rev, "Beta": z_beta}
)

plotting.plot_factor_compare(
    {"Volatility (isolated)": res_vol["factor_ret"],
     "Volatility | Mom + Rev": res_all["Volatility"],
     "Volatility | Mom + Rev + Beta": res_4["Volatility"]},
    title="Volatility: how much survives controlling for beta?",
)

In [54]:
vol_versions = pd.concat({
    "Volatility (isolated)": res_vol["factor_ret"],
    "Volatility | Mom + Rev": res_all["Volatility"],
    "Volatility | Mom + Rev + Beta": res_4["Volatility"],
}, axis=1, join="inner")

plotting.plot_table(
    factors.factor_stats(vol_versions),
    title="Volatility factor: isolated vs controlling for other factors",
    subtitle=f"{vol_versions.index.min():%b %Y} to {vol_versions.index.max():%b %Y}, common dates",
    index_name="Version",
    fmt={"Days": "{:,.0f}"},
    caption="Daily cross-sectional regressions on lagged standardized exposures; returns per 1σ of exposure.",
)

___

### Simple Dollar Volume
___

In [56]:
dvol = factors.dollar_volume(prices, volume)
z_dvol = factors.standardize(dvol)
res_dvol = factors.cs_regression(returns, z_dvol)

plotting.plot_factor_returns(res_dvol["factor_ret"], name="Dollar volume (63d)")

In [57]:
plotting.plot_cs_r2(res_dvol["r2"], name="Dollar volume (63d)")

In [58]:
plotting.plot_exposure_corr(z_dvol, z_beta, "Dollar volume", "Beta")

In [59]:
plotting.plot_exposure_corr(z_dvol, z_mom, "Dollar volume", "Momentum")

In [61]:
res_5 = factors.cs_regression_multi(returns, {
    "Momentum": z_mom, "Volatility": z_vol, "Reversal": z_rev,
    "Beta": z_beta, "Dollar volume": z_dvol,
})

dvol_versions = pd.concat({
    "Dollar volume (isolated)": res_dvol["factor_ret"],
    "Dollar volume | all other factors": res_5["Dollar volume"],
}, axis=1, join="inner")

plotting.plot_factor_compare(
    {c: dvol_versions[c] for c in dvol_versions},
    title="Dollar volume: isolated vs controlling for other factors",
)

In [62]:
plotting.plot_table(factors.factor_stats(dvol_versions),
                    title="Dollar volume factor: isolated vs joint",
                    index_name="Version", fmt={"Days": "{:,.0f}"})

In [63]:
plotting.plot_table(
    factors.factor_stats(res_5[["Momentum", "Volatility", "Reversal", "Beta", "Dollar volume"]]),
    title="Pure factor returns: five-factor model", index_name="Factor", fmt={"Days": "{:,.0f}"},
)

In [64]:
r2 = pd.DataFrame({
    "Mom + Vol + Rev": res_all["r2"],
    "+ Beta": res_4["r2"],
    "+ Dollar volume": res_5["r2"],
}).dropna().agg(["mean", "median"]).mul(100).T
r2.columns = ["Mean R² (%)", "Median R² (%)"]
plotting.plot_table(r2, title="Cross-sectional R² as factors are added", index_name="Model")

___

### Decomposing AAPL Returns (Naive, Simple Factor Model)
___

In [66]:
styles = {"Momentum": z_mom, "Volatility": z_vol, "Reversal": z_rev,
          "Beta": z_beta, "Dollar volume": z_dvol}

res_full, resid = factors.cs_regression_multi(
    returns, {**styles, **sectors}, intercept=False, return_resid=True
)

In [68]:
# 1. Sectors should add a lot of explanatory power
r2 = pd.DataFrame({"Styles only": res_5["r2"], "Styles + sectors": res_full["r2"]}).dropna()
r2 = r2.agg(["mean", "median"]).mul(100).T
r2.columns = ["Mean R² (%)", "Median R² (%)"]
plotting.plot_table(r2, title="Cross-sectional R²: adding sectors", index_name="Model")

In [72]:
X_aapl = pd.DataFrame({k: v["AAPL"] for k, v in styles.items()}).reindex(aapl.index)
plotting.plot_table(X_aapl.iloc[-1].rename("z-score"),
                    title="Apple's style exposures (latest)", index_name="Factor")

In [73]:
plotting.plot_attribution(aapl, "AAPL")

In [74]:
totals = (aapl.dropna() * 100).sum().rename("Cumulative contribution (%)")
plotting.plot_table(totals, title="AAPL: total contribution by component",
                    subtitle=f"{aapl.dropna().index.min():%b %Y} to {aapl.dropna().index.max():%b %Y}",
                    index_name="Component", fmt="{:.1f}")

### Market vs Beta vs Sector

**Market**: the move every stock shares. Every stock has exposure 1 to it.

**Beta**: stocks respond to the market unequally. The beta *factor* captures the **spread**: on an up day high-beta stocks beat low-beta ones, on a down day the reverse. Beta exposures are z-scored (mean zero across stocks), so this factor captures *who moved more or less than average*, not the market's level.

**Sector**: stocks in the same industry move together for their own reasons (oil for energy, AI news for tech), beyond the market.

**Example day**: market $+2\%$, beta factor $+0.5\%$ per $1\sigma$, tech beats the market by $1\%$.
- Typical stock: $\approx +2\%$ (market)
- Stock with beta $z = +1$: extra $+0.5\%$ (beta)
- If also a tech stock: extra $+1\%$ (sector)

### Why sector dummies absorb the market

Every stock is in exactly one sector, so for every stock the dummies sum to 1:

$$
\sum_s D_{i,s} = 1 \quad \text{for all } i
$$

That sum is identical to an intercept column (a 1 for every stock). So any move common to all stocks, i.e. the market, can be expressed by adding the same amount to every sector's return:

$$
\sum_s D_{i,s} \, (f_s + m) = \sum_s D_{i,s} \, f_s + m
$$

The regression can't tell "the market rose $m$" apart from "every sector rose $m$". With no separate market factor, each sector return therefore carries the market move plus that sector's own move.

### How Apple's time-series beta ($\approx 1.1$) splits in this model

- The **"1"** comes through the sector dummy, which carries the market one-for-one
- The **"0.1 extra"** comes through Apple's positive beta z-score $\times$ the beta factor return

So Apple's "Sector" contribution is really market + tech, and "Beta" is only the *excess* market sensitivity relative to the average stock.

**Separating them:** add an explicit market factor (exposure 1 for every stock) and constrain sector returns to net to zero (e.g. cap-weighted). Then market, sector and beta each get their own clean line.

In [76]:
w = aapl.dropna()
w = w[w.index >= w.index.max() - pd.DateOffset(years=1)]
idx, contrib = w.index, w * 100
names = list(styles)

expo = pd.Series({k: styles[k]["AAPL"].shift(1).reindex(idx).mean() for k in names})
expo["Sector"] = 1.0

fret = res_full.loc[idx, names].sum() * 100
fret["Sector"] = res_full.loc[idx, "Information Technology"].sum() * 100

tbl = pd.DataFrame({
    "Avg exposure": expo,
    "Factor return (%)": fret,
    "Contribution (%)": contrib.sum(),
    "Ann. vol (%)": contrib.std() * np.sqrt(252),
    "Sharpe": contrib.mean() / contrib.std() * np.sqrt(252),
}).reindex([*names, "Sector", "Idio", "Total"])

plotting.plot_table(
    tbl, title="AAPL: one-year factor breakdown",
    subtitle=f"{idx.min():%b %d, %Y} to {idx.max():%b %d, %Y}",
    index_name="Component",
    caption="Contribution = sum of daily lagged exposure × factor return. "
            "Sector = Information Technology (includes the market).",
)

### Risk Decomposition: Where Does a Stock's Risk Come From?

The return attribution asks *where did the return come from?* The risk decomposition asks **where did the variance come from?** For any position, it answers how much risk is factor exposure versus stock-specific.

**Problem:** components are correlated (e.g. beta and vol), so their variances don't simply add.

**Fix:** total return is exactly the sum of its components, $r = \sum_j c_j$, so its variance splits exactly into covariances:

$$
\mathrm{Var}(r) = \mathrm{Cov}\Big(\sum_j c_j, \, r\Big) = \sum_j \mathrm{Cov}(c_j, r)
$$

Each component's **share of risk**:

$$
\text{share}_j = \frac{\mathrm{Cov}(c_j, r)}{\mathrm{Var}(r)} = \frac{\sigma_j \, \rho_{j,r}}{\sigma_r}
$$

- $\sigma_j$: vol of component $j$
- $\rho_{j,r}$: its correlation with the total return
- $\sigma_r$: total vol

Shares sum to exactly 100%. Correlation matters as much as size:
- A volatile component uncorrelated with the total contributes little
- A component moving *against* the total gets a **negative** share: it hedges the rest

### Ex-post vs ex-ante

**Ex-post (realized):** decompose what actually happened, using historical contributions. Descriptive: where did risk come from?

**Ex-ante (forecast):** combine the stock's *current* exposures $x$ with a forecast factor covariance matrix $F$ and idio variance $\sigma^2_\varepsilon$:

$$
\sigma^2_{\text{forecast}} = x^\top F \, x + \sigma^2_\varepsilon
$$

This is what a pod's risk system reports: predicted risk of a position going forward, split into factor risk ($x^\top F x$) and idio risk ($\sigma^2_\varepsilon$).

In [77]:
rd = factors.risk_decomp(w)

factor_rows = [*names, "Sector"]
summary = pd.Series({
    "All factors": rd.loc[factor_rows, "Share of variance (%)"].sum(),
    "Idio": rd.loc["Idio", "Share of variance (%)"],
}).rename("Share of variance (%)")

plotting.plot_table(rd, title="AAPL: one-year risk decomposition",
                    subtitle=f"{w.index.min():%b %d, %Y} to {w.index.max():%b %d, %Y}",
                    index_name="Component",
                    caption="Share = Cov(component, total) / Var(total) = vol × corr / total vol. Shares sum to 100%.")
plotting.plot_table(summary, title="AAPL: factor vs idio risk", index_name="")

### How a Pod's Risk System Forecasts Risk (Ex-Ante)

The model is re-estimated **every day**, in four pieces:

**1. Exposures today: $x_i$**
Each stock's current style z-scores (momentum, vol, beta, ...) and industry dummies. These are known today, so no forecasting is needed.

**2. Factor covariance: $F$**
From the history of daily factor returns (one per factor, from the daily cross-sectional regressions), estimate how factors vary and co-vary:
- **Exponential weighting:** recent days count more, so the forecast adapts to the current regime. Vol typically uses a shorter half-life than correlations.
- **Adjustments:** for autocorrelation in daily returns, for vol regime shifts, and to fix known biases in sample covariance matrices.

**3. Specific (idio) risk: $\sigma^2_{\varepsilon,i}$**
From each stock's history of residuals $\varepsilon_{i,t}$, again exponentially weighted. Single-stock estimates are noisy, so they are **shrunk toward similar stocks** (same size, industry, vol level).

**4. Combine for a position or portfolio**
For a single stock:

$$
\sigma^2_i = x_i^\top F \, x_i + \sigma^2_{\varepsilon,i}
$$

For a portfolio with weights $w_i$, aggregate exposures first, $X_p = \sum_i w_i x_i$:

$$
\sigma^2_p = \underbrace{X_p^\top F \, X_p}_{\text{factor risk}} + \underbrace{\sum_i w_i^2 \, \sigma^2_{\varepsilon,i}}_{\text{idio risk}}
$$

**Key asymmetry:** idio risk is assumed uncorrelated across stocks, so it **diversifies away** as the book grows ($w_i^2$ terms shrink). Factor risk **doesn't**: exposures add up across positions. This is why pods cap factor exposures: a book of 50 "different" stocks can secretly carry one large factor bet.

**Outputs a PM sees daily:** total forecast vol, the factor vs idio split, each factor's contribution to risk, and exposures against limits.

___

### Risk Decomposition: Covariance and Correlation from First Principles
___

### Why Correlations Matter for Risk: First Principles

### Two components

Start with a return made of two pieces, $r = a + b$. Its variance is:

$$
\mathrm{Var}(r) = \mathrm{Var}(a) + \mathrm{Var}(b) + 2\,\mathrm{Cov}(a, b)
$$

The cross term decides everything:
- **Positive covariance:** the pieces move together, so their swings **reinforce** and total risk is bigger than either alone suggests
- **Zero covariance:** swings partly offset by chance, so risk grows more slowly than the pieces' vols added up
- **Negative covariance:** the pieces move against each other, so swings **cancel** and one hedges the other

With vols of 10 each:

| Correlation | Total variance | Total vol |
|---|---|---|
| $+1$ | $100 + 100 + 200 = 400$ | 20 |
| $0$ | $100 + 100 + 0 = 200$ | 14.1 |
| $-1$ | $100 + 100 - 200 = 0$ | 0 |

Same two components, very different total risk, purely from correlation.

### Allocating risk to each piece

To say how much risk belongs to each piece, split the cross term evenly between the pair. Each piece's share is its covariance with the total:

$$
\mathrm{Cov}(a, r) = \underbrace{\mathrm{Var}(a)}_{\text{own risk}} + \underbrace{\mathrm{Cov}(a, b)}_{\text{interaction}}
$$

The shares add up to the total exactly: $\mathrm{Cov}(a, r) + \mathrm{Cov}(b, r) = \mathrm{Var}(r)$. In general, with many components:

$$
\mathrm{Cov}(c_j, r) = \mathrm{Var}(c_j) + \sum_{k \neq j} \mathrm{Cov}(c_j, c_k)
$$

Each component's risk is its **own variance plus how it interacts with everything else**.

### Negative shares: hedges

Example: $a$ has vol 20, $b$ has vol 5, correlation $-0.8$, so $\mathrm{Cov}(a, b) = -80$.

$$
\mathrm{Var}(r) = 400 + 25 - 160 = 265
$$

| Piece | Own var | + Interaction | = Cov with total | Share |
|---|---|---|---|---|
| $a$ | 400 | $-80$ | 320 | 121% |
| $b$ | 25 | $-80$ | $-55$ | $-21\%$ |

$b$'s interaction outweighs its own variance, so its share is **negative**: it reduces total risk. That's what a hedge is. Shares still sum to 100%, and $a$ shows over 100% because it carries more than all the risk on its own.

**Takeaway:** a component's risk depends not just on how volatile it is, but on whether it moves with or against everything else.

___

### Incorporating Market vs Sector Split
___

In [78]:
res_mkt, resid_mkt = factors.cs_regression_market(returns, styles, sectors, return_resid=True)

In [83]:
market = pd.DataFrame(1.0, index=returns.index, columns=returns.columns)
exposures_mkt = {"Market": market, **sectors, **styles}

aapl2 = factors.attribute("AAPL", returns, exposures_mkt, res_mkt, resid_mkt,
                          groups={"Sector": list(sectors)})

plotting.plot_attribution(aapl2, "AAPL")

In [84]:
totals = (aapl2.dropna() * 100).sum().rename("Cumulative contribution (%)")
plotting.plot_table(totals, title="AAPL: total contribution, market split out",
                    index_name="Component", fmt="{:.1f}")

In [86]:
w2 = aapl2.dropna()
w2 = w2[w2.index >= w2.index.max() - pd.DateOffset(years=1)]
plotting.plot_table(factors.risk_decomp(w2), title="AAPL: one-year risk decomposition (market split)",
                    index_name="Component")

___

### Fundamental Factor Incorporation
___